In [25]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('train_data_cleaned.csv')
test_df = pd.read_csv('test_data_cleaned.csv')



In [26]:
X_train = train_df.drop('Price', axis=1)
y_train = train_df['Price']

X_test = test_df.drop('Price', axis=1)
y_test = test_df['Price']


In [27]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify categorical and numerical columns
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)



In [28]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

ridge_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('ridge', Ridge())
])



In [29]:
ridge_model.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['ID', 'Levy', 'Prod. year', 'Mileage', 'Cylinders', 'Airbags'], dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['Manufacturer', 'Model', 'Category', 'Leather interior', 'Fuel type',
       'Engine volume', 'Gear box type', 'Drive wheels', 'Doors', 'Wheel',
       'Color'],
      dtype='object'))])),
                ('ridge', Ridge())])

In [30]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred = ridge_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse:.2f}")
print(f"R2: {r2:.2f}")


MSE: 2328412674.76
R2: -3.99


In [31]:
from sklearn.model_selection import GridSearchCV

param_grid = {'ridge__alpha': [0.1, 1.0, 10.0, 50.0, 100.0]}
grid_search = GridSearchCV(ridge_model, param_grid, cv=5, scoring='r2')
grid_search.fit(X_train, y_train)

print(f"Best alpha: {grid_search.best_params_['ridge__alpha']}")


Best alpha: 100.0


In [32]:
final_model = grid_search.best_estimator_
final_predictions = final_model.predict(X_test)


In [33]:
y_pred = ridge_model.predict(X_test)


In [34]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R² Score: {r2:.2f}")


Mean Squared Error (MSE): 2328412674.76
R² Score: -3.99


In [36]:
import pandas as pd
from google.colab import files

# Make sure to use the predictions from your tuned model
# final_predictions is from the GridSearchCV best model
pred_df = pd.DataFrame({
    'Actual_Price': y_test,
    'Predicted_Price': final_predictions
})

# Save predictions to CSV
pred_csv_filename = 'predictions.csv'
pred_df.to_csv(pred_csv_filename, index=False)

# Download the CSV
files.download(pred_csv_filename)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>